# Potencial de Oleaje

**Materia:** Energías del Océano  
**Alumna:** Montserrat González  
**Fecha:** Marzo 2026  

---

## 1. Introducción

La energía undimotriz — aquella contenida en el oleaje oceánico — representa una de las fuentes renovables con mayor densidad energética por unidad de frente de ola. Su aprovechamiento requiere, como primer paso, la **caracterización del recurso** en un sitio de interés.

En este trabajo se analiza un año completo (2018) de registros espectrales de oleaje obtenidos de una boya oceánica del sistema NDBC (National Data Buoy Center, NOAA). A partir de la densidad espectral de energía $S(f)$ se calculan los parámetros de oleaje y la potencia disponible siguiendo la metodología de **Silva Casarín R. (2005, pp. 61-65)**.

### Objetivos

1. Calcular los parámetros espectrales del oleaje ($H_s$, $T_e$, $F_p$, $P_w$) a partir de los momentos del espectro.
2. Graficar la serie temporal de potencia del oleaje.
3. Determinar la potencia media anual y los promedios mensuales.
4. Construir diagramas de dispersión $H_s$–$T_e$ y $H_s$–$F_p$.
5. Estimar el recurso energético anual (GWh/m/año) mediante dos métodos:
   - **Método 1:** Sumatoria directa de potencia horaria.
   - **Método 2:** Matriz de dispersión $H_s$–$T_e$.

## 2. Marco teórico

### Espectro de oleaje

El estado del mar en un punto se describe mediante la **densidad espectral de energía** $S(f)$, que representa la distribución de la energía del oleaje en función de la frecuencia $f$ (Hz). En la práctica, los espectrómetros de las boyas proporcionan $S(f)$ discretizado en $N$ bandas de frecuencia, cada una con un ancho $\Delta f_i$.

### Momentos espectrales

A partir del espectro discreto se calculan los **momentos espectrales** de orden $n$:

$$m_n = \sum_{i=1}^{N} f_i^n \cdot S(f_i) \cdot \Delta f_i$$

Los momentos de interés son:

| Momento | Expresión | Significado físico |
|---------|-----------|-------------------|
| $m_0$ | $\sum S(f_i) \cdot \Delta f_i$ | Varianza total de la superficie = energía total del espectro |
| $m_{-1}$ | $\sum \frac{S(f_i)}{f_i} \cdot \Delta f_i$ | Ponderación hacia bajas frecuencias (relacionado con transporte de energía) |
| $m_1$ | $\sum f_i \cdot S(f_i) \cdot \Delta f_i$ | Frecuencia media del espectro |

### Parámetros derivados

| Parámetro | Fórmula | Descripción |
|-----------|---------|-------------|
| **Altura significante** | $H_s = 4\sqrt{m_0}$ | Medida estadística de la altura de ola (promedio del tercio superior) |
| **Período de energía** | $T_e = m_{-1} / m_0$ | Período asociado al transporte de energía del oleaje |
| **Frecuencia pico** | $F_p = f$ en $\max[S(f)]$ | Frecuencia donde el espectro alcanza su máximo |

### Potencia del oleaje

La **densidad de flujo de energía** (potencia por metro de frente de ola) se obtiene de la teoría lineal de oleaje en aguas profundas:

$$P_w = \frac{\rho g^2}{64\pi} H_s^2 T_e$$

donde $\rho = 1025$ kg/m³ (densidad del agua de mar) y $g = 9.81$ m/s². El coeficiente resultante es:

$$\frac{\rho g^2}{64\pi} = \frac{1025 \times 9.81^2}{64\pi} \approx 490.6 \text{ W/m}$$

Por lo tanto: $P_w \approx 0.4906 \cdot H_s^2 \cdot T_e$ (en kW/m).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)


## 3. Carga y preparación de datos

Los datos provienen de una boya oceánica del NDBC y fueron descargados como archivo CSV. El archivo contiene **6,467 observaciones horarias** correspondientes al año 2018, con la siguiente estructura:

- **Columnas temporales:** Año, mes, día, hora, minuto.
- **47 columnas de frecuencia** (de 0.02 a 0.485 Hz): cada una contiene la densidad espectral $S(f_i)$ en m²/Hz.
- **Fila especial (fila 2 del CSV):** Contiene los anchos de banda $\Delta f_i$ para cada bin de frecuencia. Estos anchos **no son uniformes** (varían de 0.005 a 0.02 Hz).
- **Columnas pre-calculadas:** $H_s$, $T_e$, $F_p$, $P_w$ (se usarán para validación).

In [ ]:
# Cargar archivo
datos_raw = pd.read_csv("data/Oleaje_Datos.csv")
datos_raw.columns = datos_raw.columns.str.strip()

# Identificar columnas de frecuencia (nombres parseables como float entre 0 y 1)
freq_cols = []
for col in datos_raw.columns:
    try:
        val = float(col)
        if 0 < val < 1:
            freq_cols.append(col)
    except ValueError:
        pass

# Extraer frecuencias centrales y anchos de banda (Δf) de la fila 0
frecuencias = np.array([float(c) for c in freq_cols])
delta_f = datos_raw.loc[0, freq_cols].values.astype(float)

print(f"Bandas de frecuencia: {len(frecuencias)} bins ({frecuencias[0]} - {frecuencias[-1]} Hz)")
print(f"Δf (anchos de banda): {delta_f}")

# Eliminar fila de Δf (índice 0) y quedarse solo con datos reales
df = datos_raw.iloc[1:].copy()

# Eliminar columnas artefacto
cols_drop = [c for c in df.columns if "Unnamed" in str(c) or c.strip() == ""]
df = df.drop(columns=cols_drop, errors="ignore")

# Columnas numéricas de frecuencia
for col in freq_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"\nObservaciones cargadas: {len(df)}")

### Construcción de la serie temporal

Se construyen las columnas temporales a partir de las columnas de año, mes, día y hora para generar un índice `datetime` que permita el análisis temporal y las agrupaciones por mes.

In [ ]:
# Columnas de tiempo
cols_time = ["AÑO", "MES", "DIA", "HR"]

# Limpiar strings y espacios
for col in cols_time:
    df[col] = df[col].astype(str).str.strip()

# Convertir a numérico (manejo de errores)
for col in cols_time:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Eliminar filas inválidas
df = df.dropna(subset=cols_time)

# Convertir a enteros
for col in cols_time:
    df[col] = df[col].astype(int)


In [ ]:
df["datetime"] = pd.to_datetime(
    df["AÑO"].astype(str)
    + "-"
    + df["MES"].astype(str)
    + "-"
    + df["DIA"].astype(str)
    + " "
    + df["HR"].astype(str)
    + ":00"
)

# Ordenar
df = df.sort_values("datetime")

df.head()


,AÑO,MES,DIA,HR,MN,Fecha / Hora,0.02,0.0325,0.0375,0.0425,0.0475,0.0525,0.0575,0.0625,0.0675,0.0725,0.0775,0.0825,0.0875,0.0925,0.1,0.11,0.12,0.13,0.14,0.15,0.16,0.17,0.18,0.19,0.2,0.21,0.22,0.23,0.24,0.25,0.26,0.27,0.28,0.29,0.3,0.31,0.32,0.33,0.34,0.35,0.365,0.385,0.405,0.425,0.445,0.465,0.485,Hs (m),Te (s),Fp (Hz),Pw (kW/m),Pw prom anual (kW/m),Unnamed: 58,,datetime
1,2018,1,1,0,40.0,01/01/2018 00:40,0.0,0.0,0.0,0.0,0.0,0.08,0.27,3.57,6.84,6.21,5.41,3.43,1.68,1.21,0.89,0.78,0.71,0.65,0.45,0.52,0.57,0.55,0.75,0.61,0.27,0.29,0.13,0.12,0.09,0.07,0.08,0.07,0.08,0.11,0.04,0.05,0.05,0.05,0.07,0.04,0.01,0.01,0.01,0.01,0.01,0.01,0.01,1.902314,11.153682,0.0675,19.802264,29.845227,NaN,NaN,2018-01-01 00:00:00
2,2018,1,1,1,40.0,01/01/2018 01:40,0.0,0.0,0.0,0.0,0.0,0.00,0.25,6.01,9.79,10.37,7.72,3.29,3.39,1.81,0.70,0.45,0.99,0.82,0.58,0.75,0.35,0.54,0.59,0.24,0.21,0.19,0.17,0.24,0.14,0.09,0.09,0.06,0.06,0.06,0.07,0.05,0.05,0.04,0.03,0.03,0.02,0.02,0.01,0.02,0.02,0.01,0.01,2.163562,11.844518,0.0725,27.201199,NaN,NaN,NaN,2018-01-01 01:00:00
3,2018,1,1,2,40.0,01/01/2018 02:40,0.0,0.0,0.0,0.0,0.0,0.09,0.35,2.82,4.06,7.33,7.44,2.71,2.09,1.42,0.46,0.72,0.95,0.48,0.66,0.48,1.04,0.46,0.20,0.24,0.42,0.20,0.26,0.17,0.12,0.10,0.12,0.06,0.07,0.05,0.03,0.05,0.03,0.03,0.03,0.03,0.02,0.02,0.03,0.01,0.01,0.01,0.01,1.873926,11.064877,0.0775,19.062666,NaN,NaN,NaN,2018-01-01 02:00:00
4,2018,1,1,3,40.0,01/01/2018 03:40,0.0,0.0,0.0,0.0,0.0,0.03,0.49,2.96,9.90,11.43,5.16,2.76,2.79,1.72,1.15,0.90,0.99,0.76,0.56,0.40,0.59,0.80,0.42,0.36,0.22,0.19,0.11,0.17,0.09,0.12,0.10,0.08,0.03,0.10,0.04,0.07,0.05,0.05,0.02,0.04,0.05,0.02,0.03,0.02,0.01,0.01,0.00,2.092988,11.555399,0.0725,24.834222,NaN,NaN,NaN,2018-01-01 03:00:00
5,2018,1,1,4,40.0,01/01/2018 04:40,0.0,0.0,0.0,0.0,0.0,0.08,0.34,4.17,11.05,9.81,3.62,2.11,1.52,0.79,0.83,0.44,0.54,0.80,0.49,1.12,0.74,0.50,0.49,0.42,0.27,0.24,0.18,0.07,0.12,0.14,0.08,0.06,0.06,0.06,0.07,0.04,0.04,0.03,0.03,0.04,0.02,0.02,0.01,0.01,0.01,0.02,0.01,1.993389,11.637500,0.0675,22.686938,NaN,NaN,NaN,2018-01-01 04:00:00


## 4. Cálculo de parámetros espectrales (Punto 4)

A partir del espectro discreto $S(f_i)$ y los anchos de banda $\Delta f_i$ extraídos del archivo, se calculan los momentos espectrales para cada observación horaria. Con estos momentos se obtienen los parámetros de oleaje: $H_s$, $T_e$, $F_p$ y $P_w$.

Este cálculo se realiza de forma vectorizada sobre las 6,467 observaciones simultáneamente.

In [ ]:
# Matriz de densidad espectral: (N_obs, 47)
S = df[freq_cols].values.astype(float)

# --- Momentos espectrales ---
m0     = np.sum(S * delta_f, axis=1)                   # m₀ = Σ S(fᵢ)·Δfᵢ
m_neg1 = np.sum((S / frecuencias) * delta_f, axis=1)   # m₋₁ = Σ (S(fᵢ)/fᵢ)·Δfᵢ
m1     = np.sum(S * frecuencias * delta_f, axis=1)      # m₁ = Σ fᵢ·S(fᵢ)·Δfᵢ

# --- Parámetros derivados ---
Hs_calc = 4.0 * np.sqrt(m0)                            # Altura significante
Te_calc = m_neg1 / m0                                   # Período de energía
Fp_calc = frecuencias[np.argmax(S, axis=1)]             # Frecuencia pico

# --- Potencia del oleaje (kW/m) ---
rho = 1025       # kg/m³
g   = 9.81       # m/s²
coeff = rho * g**2 / (64 * np.pi)                       # ≈ 490.6 W/m
Pw_calc = coeff * Hs_calc**2 * Te_calc / 1000           # kW/m

# Almacenar en el DataFrame
df["Hs_calc"] = Hs_calc
df["Te_calc"] = Te_calc
df["Fp_calc"] = Fp_calc
df["Pw_calc"] = Pw_calc

print(f"Coeficiente ρg²/(64π) = {coeff:.4f} W/m")
print(f"Cálculos completados para {len(df)} observaciones.")

### Tabla de resultados espectrales

Se muestran las primeras 20 observaciones con los parámetros calculados a partir de los momentos espectrales.

In [ ]:
tabla = df[["datetime", "Hs_calc", "Te_calc", "Fp_calc", "Pw_calc"]].copy()
tabla.columns = ["Fecha/Hora", "Hs (m)", "Te (s)", "Fp (Hz)", "Pw (kW/m)"]
tabla.head(20)

### Validación contra valores pre-calculados

El archivo CSV incluye valores de $H_s$, $T_e$, $F_p$ y $P_w$ pre-calculados. Se comparan con los valores obtenidos mediante momentos espectrales para verificar la correcta implementación del método.

In [ ]:
# Convertir columnas pre-calculadas a numérico
for col in ["Hs (m)", "Te (s)", "Fp (Hz)", "Pw (kW/m)"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("=== Validación: valores calculados vs pre-calculados ===\n")

validaciones = [
    ("Hs (m)",    "Hs_calc", "Hs (m)"),
    ("Te (s)",    "Te_calc", "Te (s)"),
    ("Fp (Hz)",   "Fp_calc", "Fp (Hz)"),
    ("Pw (kW/m)", "Pw_calc", "Pw (kW/m)"),
]

for label, calc_col, ref_col in validaciones:
    diff = (df[calc_col] - df[ref_col]).abs()
    pct  = (diff / df[ref_col].abs() * 100)
    print(f"{label}:")
    print(f"  Error absoluto medio:   {diff.mean():.8f}")
    print(f"  Error porcentual medio: {pct.mean():.6f}%")
    print(f"  Error absoluto máximo:  {diff.max():.8f}")
    print()

### Selección de variables para el análisis

Una vez validados los cálculos espectrales, se utilizan los **valores calculados** ($H_s$, $T_e$, $F_p$, $P_w$) para el resto del análisis.

In [ ]:
df = df[["datetime", "Hs_calc", "Te_calc", "Fp_calc", "Pw_calc"]].copy()
df.columns = ["datetime", "Hs (m)", "Te (s)", "Fp (Hz)", "Pw (kW/m)"]

## 5. Serie temporal de potencia del oleaje (Punto 5)

La gráfica de $P_w$ a resolución horaria permite observar la **variabilidad del recurso energético** a lo largo del año. Se esperan fluctuaciones significativas asociadas a la estacionalidad (tormentas invernales vs. calma veraniega) y a eventos individuales de alta energía.

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df["datetime"], df["Pw (kW/m)"], linewidth=0.5, color="steelblue", alpha=0.8)
ax.set_xlabel("Fecha")
ax.set_ylabel("Pw (kW/m)")
ax.set_title("Serie temporal de potencia del oleaje — 2018")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Promedios mensuales de Pw (Punto 9)

La distribución mensual de la potencia permite identificar la **estacionalidad del recurso**. Esto es fundamental para la planificación energética: un recurso con alta variabilidad estacional requiere complementarse con otras fuentes o con sistemas de almacenamiento.

In [ ]:
df["mes"] = df["datetime"].dt.month

promedio_mensual = df.groupby("mes")["Pw (kW/m)"].mean()

meses_nombres = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
promedio_mensual.index = meses_nombres

promedio_anual = df["Pw (kW/m)"].mean()

print("Promedios mensuales de Pw (kW/m):\n")
for mes, val in zip(meses_nombres, promedio_mensual.values):
    print(f"  {mes}: {val:.2f} kW/m")
print(f"\n  Promedio anual: {promedio_anual:.2f} kW/m")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(1, 13), promedio_mensual.values, color="steelblue", edgecolor="navy")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_nombres)
ax.set_xlabel("Mes")
ax.set_ylabel("Pw promedio (kW/m)")
ax.set_title("Potencia promedio mensual del oleaje — 2018")
ax.axhline(y=promedio_anual, color="red", linestyle="--", linewidth=1.5,
           label=f"Promedio anual: {promedio_anual:.1f} kW/m")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# Etiquetas de valor sobre cada barra
for bar, val in zip(bars, promedio_mensual.values):
    ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height() + 0.5,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Potencia promedio anual (Punto 6)

La potencia media anual es el indicador principal para evaluar la **viabilidad energética** de un sitio. Como referencia, sitios con $P_w > 20$ kW/m se consideran de interés para el aprovechamiento de energía del oleaje.

In [ ]:
print(f"Potencia promedio anual: {promedio_anual:.2f} kW/m")

## 8. Recurso energético anual — Método 1 (Punto 10)

La energía anual disponible por metro de frente de ola se obtiene integrando la potencia en el tiempo. Con datos horarios, la integral se aproxima como:

$$E = \sum_{i=1}^{N} P_{w,i} \cdot \Delta t \quad \text{donde } \Delta t = 1 \text{ hora}$$

Se presentan dos cálculos:
- **Sumatoria directa:** suma de las $N$ observaciones reales disponibles.
- **Extrapolación anual:** asumiendo que el promedio calculado es representativo de las 8,760 horas del año.

In [ ]:
# Método 1a: Sumatoria directa (Δt = 1 hora por observación)
N_obs = len(df)
E_directa_kWh = df["Pw (kW/m)"].sum() * 1  # kWh/m
E_directa_MWh = E_directa_kWh / 1000
E_directa_GWh = E_directa_kWh / 1e6

# Método 1b: Extrapolación a año completo (8760 horas)
horas_anio = 8760
E_extrap_kWh = promedio_anual * horas_anio
E_extrap_MWh = E_extrap_kWh / 1000
E_extrap_GWh = E_extrap_kWh / 1e6

print("=== Recurso energético anual (Método 1) ===\n")
print(f"Observaciones disponibles: {N_obs} horas (de {horas_anio} en el año)")
print(f"Cobertura temporal: {N_obs/horas_anio*100:.1f}%\n")

print("--- Sumatoria directa (datos reales) ---")
print(f"  E = {E_directa_kWh:,.2f} kWh/m")
print(f"  E = {E_directa_MWh:,.2f} MWh/m")
print(f"  E = {E_directa_GWh:,.4f} GWh/m\n")

print("--- Extrapolación a año completo (Pw_prom × 8760 h) ---")
print(f"  E = {E_extrap_kWh:,.2f} kWh/m")
print(f"  E = {E_extrap_MWh:,.2f} MWh/m")
print(f"  E = {E_extrap_GWh:,.4f} GWh/m")

## 9. Diagramas de dispersión (Punto 7)

Los diagramas de dispersión permiten visualizar las **combinaciones de estados de mar** ($H_s$–$T_e$ y $H_s$–$F_p$) coloreadas por la potencia $P_w$.

- En el diagrama $H_s$–$T_e$: se observa que la mayor potencia se concentra en olas altas con períodos largos, consistente con la fórmula $P_w \propto H_s^2 T_e$.
- En el diagrama $H_s$–$F_p$: frecuencias bajas (oleaje tipo *swell*) con alturas grandes aportan la mayor energía, mientras que frecuencias altas (oleaje tipo *sea*) presentan menor potencia.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter 1: Te vs Hs, color = Pw
sc1 = axes[0].scatter(df["Te (s)"], df["Hs (m)"], c=df["Pw (kW/m)"],
                       cmap="jet", s=5, alpha=0.6)
axes[0].set_xlabel("Te (s)")
axes[0].set_ylabel("Hs (m)")
axes[0].set_title("Hs vs Te (color = Pw)")
plt.colorbar(sc1, ax=axes[0], label="Pw (kW/m)")

# Scatter 2: Fp vs Hs, color = Pw
sc2 = axes[1].scatter(df["Fp (Hz)"], df["Hs (m)"], c=df["Pw (kW/m)"],
                       cmap="jet", s=5, alpha=0.6)
axes[1].set_xlabel("Fp (Hz)")
axes[1].set_ylabel("Hs (m)")
axes[1].set_title("Hs vs Fp (color = Pw)")
plt.colorbar(sc2, ax=axes[1], label="Pw (kW/m)")

plt.tight_layout()
plt.show()

## 10. Matriz de dispersión Hs–Te y energía anual — Método 2 (Punto 11)

El **Método 2** utiliza una tabla de doble entrada (*scatter matrix*) que clasifica las observaciones por rangos de $H_s$ y $T_e$. Para cada celda $(j, k)$ de la matriz:

1. Se cuenta el número de horas de ocurrencia.
2. Se calcula la potencia usando los centros de bin: $P_w(H_{s,j}, T_{e,k})$.
3. La energía total es: $E = \sum_j \sum_k P_w(H_{s,j}, T_{e,k}) \cdot \text{horas}(j,k)$

Este método difiere del Método 1 porque usa valores discretizados (centros de bin) en lugar de los valores exactos de cada observación, lo cual introduce un **error por discretización** que depende del tamaño de los bins.

Se utilizan bins de **0.5 m** para $H_s$ y **1.0 s** para $T_e$, que es la resolución estándar en estudios de ingeniería de costas.

In [ ]:
# Definir bins
Hs_bin_width = 0.5  # metros
Te_bin_width = 1.0  # segundos

Hs_bins = np.arange(0, df["Hs (m)"].max() + Hs_bin_width, Hs_bin_width)
Te_bins = np.arange(0, df["Te (s)"].max() + Te_bin_width, Te_bin_width)

Hs_centers = (Hs_bins[:-1] + Hs_bins[1:]) / 2
Te_centers = (Te_bins[:-1] + Te_bins[1:]) / 2

# Asignar cada observación a un bin
df["Hs_bin"] = pd.cut(df["Hs (m)"], bins=Hs_bins, include_lowest=True)
df["Te_bin"] = pd.cut(df["Te (s)"], bins=Te_bins, include_lowest=True)

# Matriz de ocurrencia: horas en cada (Hs, Te)
occurrence = df.groupby(["Hs_bin", "Te_bin"], observed=False).size().unstack(fill_value=0)

# Pw para cada combinación de centros de bin
rho = 1025
g = 9.81
coeff = rho * g**2 / (64 * np.pi)

Pw_bin = np.zeros((len(Hs_centers), len(Te_centers)))
for j, hs in enumerate(Hs_centers):
    for k, te in enumerate(Te_centers):
        Pw_bin[j, k] = coeff * hs**2 * te / 1000  # kW/m

# Energía por bin = Pw × horas
E_bin = Pw_bin * occurrence.values  # kWh/m

# Energía total (Método 2)
E_m2_kWh = E_bin.sum()
E_m2_MWh = E_m2_kWh / 1000
E_m2_GWh = E_m2_kWh / 1e6

print("=== Recurso energético anual (Método 2 — Matriz Hs-Te) ===\n")
print(f"Bins de Hs: {len(Hs_centers)} ({Hs_bins[0]:.1f} a {Hs_bins[-1]:.1f} m, ancho {Hs_bin_width} m)")
print(f"Bins de Te: {len(Te_centers)} ({Te_bins[0]:.1f} a {Te_bins[-1]:.1f} s, ancho {Te_bin_width} s)")
print(f"Total horas en matriz: {occurrence.values.sum()} (de {len(df)} observaciones)\n")
print(f"E = {E_m2_kWh:,.2f} kWh/m/año")
print(f"E = {E_m2_MWh:,.2f} MWh/m/año")
print(f"E = {E_m2_GWh:,.4f} GWh/m/año")

print(f"\n--- Comparación de métodos ---")
print(f"Método 1 (sumatoria directa): {E_directa_MWh:,.2f} MWh/m/año")
print(f"Método 2 (matriz Hs-Te):      {E_m2_MWh:,.2f} MWh/m/año")
print(f"Diferencia: {abs(E_m2_MWh - E_directa_MWh):,.2f} MWh/m/año "
      f"({abs(E_m2_MWh - E_directa_MWh) / E_directa_MWh * 100:.1f}%)")

### Visualización de la matriz de dispersión

Se presentan dos heatmaps:
- **Izquierda:** Matriz de ocurrencia (número de horas por combinación de bins).
- **Derecha:** Contribución energética por bin (kWh/m), que muestra dónde se concentra el recurso.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap de ocurrencia
im1 = axes[0].pcolormesh(Te_bins, Hs_bins, occurrence.values, cmap="YlOrRd", shading="flat")
axes[0].set_xlabel("Te (s)")
axes[0].set_ylabel("Hs (m)")
axes[0].set_title("Matriz de ocurrencia (horas)")
plt.colorbar(im1, ax=axes[0], label="Horas")

# Heatmap de contribución energética
im2 = axes[1].pcolormesh(Te_bins, Hs_bins, E_bin, cmap="YlOrRd", shading="flat")
axes[1].set_xlabel("Te (s)")
axes[1].set_ylabel("Hs (m)")
axes[1].set_title("Contribución energética por bin (kWh/m)")
plt.colorbar(im2, ax=axes[1], label="kWh/m")

plt.tight_layout()
plt.show()

## 11. Conclusiones

A partir del análisis espectral de un año completo de datos de oleaje (2018), se obtuvieron los siguientes resultados:

**Caracterización del recurso:**
- La potencia media anual del sitio es de aproximadamente **29.84 kW/m**, lo cual lo posiciona como un sitio con potencial moderado-alto para aprovechamiento de energía undimotriz.
- El recurso presenta una marcada **estacionalidad**, con los meses de invierno mostrando mayor potencia disponible.

**Cálculos espectrales:**
- Los parámetros $H_s$, $T_e$, $F_p$ y $P_w$ calculados mediante momentos espectrales coinciden con los valores pre-calculados del archivo, validando la implementación del método de Silva Casarín (2005).

**Recurso energético anual:**
- **Método 1** (sumatoria directa): proporciona el valor más preciso al usar cada observación horaria individualmente.
- **Método 2** (matriz de dispersión): introduce un pequeño error por discretización al usar centros de bin, pero tiene la ventaja de representar de forma compacta el clima de oleaje del sitio y facilitar el dimensionamiento de dispositivos WEC (*Wave Energy Converters*).

**Diagramas de dispersión:**
- La relación $H_s$–$T_e$ muestra que las condiciones de mayor potencia corresponden a oleaje con alturas significantes elevadas y períodos de energía largos, consistente con la dependencia cuadrática $P_w \propto H_s^2 T_e$.

---

## 12. Referencias

- Silva Casarín, R. (2005). *Análisis y descripción estadística del oleaje*. Serie Docencia, Instituto de Ingeniería, UNAM. pp. 61-65.
- National Data Buoy Center (NDBC), NOAA. Datos de espectro de oleaje.